# Bernstein–Vazirani algoritmus
**Cél:** Megtalálni egy ismeretlen `s` titkos bitsztringet egyetlen kvantum lekérdezéssel.
Klasszikusan `n` lekérdezés kellene (minden bitre külön).

In [ ]:
# Alap importok
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import time
import random

simulator = AerSimulator()

## Alap BV áramkör (n=3, s='101')

In [ ]:
n = 3
secret_string = '101'   # Rejtett bitvektor (s)

qc = QuantumCircuit(n+1, n)   # n bemeneti qubit + 1 segédqubit + n klasszikus bit

# 1. lépés: Minden bemeneti qubit |+⟩ állapotba (Hadamard)
for i in range(n):
    qc.h(i)

# 2. lépés: Segédqubit |−⟩ állapotba (X majd H)
qc.x(n)
qc.h(n)
qc.barrier()

# 3. lépés: Orákulum – ahol s_i=1, ott CNOT az i-edik qubitről a segédqubitre
for i in range(n):
    if secret_string[i] == '1':
        qc.cx(i, n)
qc.barrier()

# 4. lépés: Hadamard visszafelé (interferencia)
for i in range(n):
    qc.h(i)

# 5. lépés: Mérés
qc.measure(range(n), range(n))

display(qc.draw('mpl'))

In [ ]:
job = simulator.run(qc, shots=1024)
counts = job.result().get_counts()

print("Titkos string:", secret_string)
print("Mért eredmény:", counts)
# Az algoritmus mindig pontosan az s stringet méri – ezért csak 1 eredmény van
plot_histogram(counts)

## 1. Skálázhatóság vizsgálata
Megnézzük, hogy különböző `n` értékekre mennyi ideig tart a szimuláció,
és hány kaput / milyen mély az áramkör.

In [ ]:
n_values = [3, 5, 8, 12]
futasi_idok = []

print(f"{'n':<5} | {'Klasszikus Q':<13} | {'Kvantum Q':<10} | {'Kapuszám':<10} | {'Mélység':<8} | {'Idő (s)'}")
print("-" * 65)

for test_n in n_values:
    # Áramkör felépítése
    test_s = '1' * test_n
    test_qc = QuantumCircuit(test_n + 1, test_n)
    test_qc.h(range(test_n))
    test_qc.x(test_n)
    test_qc.h(test_n)
    for i in range(test_n):
        if test_s[i] == '1':
            test_qc.cx(i, test_n)
    test_qc.h(range(test_n))
    test_qc.measure(range(test_n), range(test_n))

    # Fordítás (a szimulátorhoz optimalizált forma)
    compiled = transpile(test_qc, simulator)

    # Futási idő mérése – 5 futás átlaga, hogy megbízhatóbb legyen
    idok = []
    for _ in range(5):
        start = time.time()
        simulator.run(compiled, shots=1).result()
        idok.append(time.time() - start)
    atlag_ido = sum(idok) / len(idok)
    futasi_idok.append(atlag_ido)

    print(f"{test_n:<5} | {test_n:<13} | {1:<10} | {compiled.size():<10} | {compiled.depth():<8} | {atlag_ido:.4f}")

# Grafikon
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bal: lekérdezések száma
ax1.plot(n_values, n_values, label='Klasszikus (n lekérdezés)', marker='o', linestyle='--')
ax1.plot(n_values, [1]*len(n_values), label='Kvantum (1 lekérdezés)', marker='s', linewidth=2)
ax1.set_xlabel("n (titkos string hossza)")
ax1.set_ylabel("Lekérdezések száma")
ax1.set_title("Kvantum előny")
ax1.legend()
ax1.grid(True)

# Jobb: szimulációs idő
ax2.plot(n_values, futasi_idok, marker='o', color='green')
ax2.set_xlabel("n (titkos string hossza)")
ax2.set_ylabel("Átlagos szimulációs idő (s)")
ax2.set_title("Szimulációs idő n függvényében")
ax2.grid(True)

plt.tight_layout()
plt.show()

## 2. Részleges orákulum – hibás CNOT
Mi történik, ha az orákulum véletlenszerűen **kihagyja** vagy **megfordítja** az egyik CNOT-ot?
A kihagyás azt jelenti, hogy az adott bitet nem kérdezzük le.
A megfordítás azt jelenti, hogy extra X kaput adunk hozzá – az adott bit értéke ellentétes lesz.

In [ ]:
n_p = 5
secret_p = '11111'
hiba_index = random.randint(0, n_p - 1)          # melyik biten lesz hiba
hiba_tipus = random.choice(['kihagyás', 'megfordítás'])  # milyen hiba

print(f"Eredeti titkos string : {secret_p}")
print(f"Hiba: {hiba_tipus} a(z) {hiba_index}. indexnél")

# Áramkör felépítése
qc_p = QuantumCircuit(n_p + 1, n_p)
qc_p.h(range(n_p))
qc_p.x(n_p)
qc_p.h(n_p)
qc_p.barrier()

for i in range(n_p):
    if i == hiba_index:
        if hiba_tipus == 'kihagyás':
            pass  # CNOT kihagyva – az adott bit 0 lesz az eredményben
        else:
            # Megfordítás: CNOT + extra X kapu → az adott bit ellentétes lesz
            qc_p.cx(i, n_p)
            qc_p.x(i)
    elif secret_p[i] == '1':
        qc_p.cx(i, n_p)

qc_p.barrier()
qc_p.h(range(n_p))
qc_p.measure(range(n_p), range(n_p))

counts_p = simulator.run(transpile(qc_p, simulator), shots=1024).result().get_counts()
mert = list(counts_p.keys())[0]

print(f"Várt eredmény (hibátlan): {secret_p}")
print(f"Mért eredmény (hibás)   : {mert}")
print(f"Különbség a(z) {hiba_index}. bitpozícióban látható.")
plot_histogram(counts_p)

## 3. Klasszikus vs. kvantum lekérdezésszám
**Klasszikus megoldás:** Minden bitre külön lekérdezünk egy `e_i` egységvektorral
(pl. `e_0 = 100...0`, `e_1 = 010...0`, stb.). Az orákulum visszaadja `s · e_i mod 2 = s_i`.
Összesen `n` lekérdezés szükséges.

**Kvantum megoldás:** Egyetlen lekérdezés elég.

In [ ]:
def orakulum(x, secret):
    """Skalárszorzat mod 2: visszaadja x·s értékét."""
    return sum(int(x[i]) * int(secret[i]) for i in range(len(x))) % 2

def klasszikus_bv(secret):
    """Klasszikus megoldás: n egységvektorral kérdezi le az orákulumot."""
    n = len(secret)
    lekerdezesek = 0
    talalt = ""
    for i in range(n):
        e_i = '0' * i + '1' + '0' * (n - i - 1)  # i-edik egységvektor
        talalt += str(orakulum(e_i, secret))        # az eredmény = s_i
        lekerdezesek += 1
    return lekerdezesek, talalt

# Tesztelés n = 1..15
n_range = list(range(1, 16))
c_lekerd = []
q_lekerd = []

print(f"{'n':<6} | {'Klasszikus lekérdezés':<22} | {'Kvantum lekérdezés'}")
print("-" * 50)
for n in n_range:
    s = '1' * n
    db, talalt = klasszikus_bv(s)
    c_lekerd.append(db)
    q_lekerd.append(1)
    if n in [1, 5, 10, 15]:
        print(f"{n:<6} | {db:<22} | 1")

# Grafikon
plt.figure(figsize=(8, 4))
plt.plot(n_range, c_lekerd, label='Klasszikus (n lekérdezés)', marker='o', linestyle='--')
plt.plot(n_range, q_lekerd, label='Kvantum (1 lekérdezés)', marker='s', linewidth=2)
plt.xlabel("n (titkos string hossza)")
plt.ylabel("Lekérdezések száma")
plt.title("Klasszikus vs. kvantum – BV algoritmus")
plt.legend()
plt.grid(True)
plt.show()